In [ ]:
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

from openai import OpenAI

client = OpenAI(api_key="")

c:\Users\vinee\Desktop\AI research Engineer SHL\3MarEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open("processed_assessments_clean.json","r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Assessments:", len(df))
df.head()

Assessments: 377


,name,url,assessed_skills_norm,cognitive_dimensions_norm,target_roles_norm,industry_tags,final_duration,experience_numeric,embedding
0,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,"[portability, application development, perform...",[],"[programmers, application developers]",[Information Technology],30.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.019559728, -0.0246569775, 0.0208423324, -0..."
1,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,[],[],[],[],NaN,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[0.0381163061, 0.0718954131, 0.0741915777, 0.0..."
2,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,"[programming knowledge, validation, routing, s...",[],"[full stack developer, software engineer, tech...",[Information Technology],17.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.0091075459, -0.0168671068, 0.0248628519, -..."
3,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,"[wpf data bindings, mvvm pattern, events, data...",[],"[full stack developer, net developer, software...",[Information Technology],5.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.0426559374, 0.017746916, 0.0271624029, -0...."
4,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,"[behaviors, messages, clients, services and co...",[],"[full stack developer, net developer - wcf, so...",[Information Technology],11.0,"{'min_years': 3, 'max_years': 7, 'midpoint': 5.0}","[-0.0292214733, -0.0284445733, 0.0452607013, -..."


In [3]:
def build_retrieval_text(row):

    parts = []

    if row.get("description"):
        parts.append(row["description"])

    if isinstance(row.get("assessed_skills_norm"), list):
        parts.append("skills " + " ".join(row["assessed_skills_norm"]))

    if isinstance(row.get("target_roles_norm"), list):
        parts.append("roles " + " ".join(row["target_roles_norm"]))

    if isinstance(row.get("cognitive_dimensions_norm"), list):
        parts.append("competencies " + " ".join(row["cognitive_dimensions_norm"]))

    if isinstance(row.get("industry_tags"), list):
        parts.append("industry " + " ".join(row["industry_tags"]))

    return " ".join(parts)

df["retrieval_text"] = df.apply(build_retrieval_text, axis=1)

In [4]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

c:\Users\vinee\Desktop\AI research Engineer SHL\3MarEnv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

c:\Users\vinee\Desktop\AI research Engineer SHL\3MarEnv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
embeddings = model.encode(
    df["retrieval_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 12/12 [00:04<00:00,  2.89it/s]


In [7]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_features=50000
)

tfidf_matrix = tfidf.fit_transform(df["retrieval_text"])
tfidf_matrix = normalize(tfidf_matrix)

In [8]:
def reason_query(query):

    prompt = f"""
You are an expert HR assessment consultant.

Analyze the hiring query or job description.

Infer what assessments should evaluate.

Return JSON with fields:

role
seniority
experience_years
technical_skills
implied_skills
competencies
assessment_focus

technical_skills:
explicit technologies mentioned.

implied_skills:
technologies or tools commonly associated with the role.

competencies:
human competencies to evaluate.

assessment_focus:
types of assessments required.

Query:
{query}

Return JSON only.
"""

    res = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role":"user","content":prompt}]
    )

    txt = res.choices[0].message.content

    m = re.search(r"\{.*\}", txt, re.DOTALL)

    if m:
        return json.loads(m.group())

    return {}

In [9]:
def detect_intent(parsed):

    if parsed.get("technical_skills"):
        return "skill"

    if parsed.get("role"):
        return "role"

    if parsed.get("competencies"):
        return "competency"

    return "general"

In [23]:
def flatten_list(items):

    if not items:
        return []

    flattened = []

    for x in items:

        if isinstance(x, dict):
            flattened.extend([str(v) for v in x.values()])

        elif isinstance(x, list):
            flattened.extend(flatten_list(x))

        else:
            flattened.append(str(x))

    return flattened


def build_query_text(parsed):

    parts = []

    parts.append(str(parsed.get("role","")))

    parts += flatten_list(parsed.get("technical_skills",[]))
    parts += flatten_list(parsed.get("implied_skills",[]))
    parts += flatten_list(parsed.get("competencies",[]))
    parts += flatten_list(parsed.get("assessment_focus",[]))

    # remove empty strings
    parts = [p for p in parts if p.strip()]

    return " ".join(parts)

In [11]:
def retrieve_candidates(query_text, top_k=150):

    q_emb = model.encode(query_text, normalize_embeddings=True)

    sims = cosine_similarity([q_emb], embeddings)[0]

    idx = np.argsort(sims)[::-1][:top_k]

    return idx, sims

In [12]:
def adaptive_score(parsed, intent, idx, sims, query_text):

    q_tfidf = tfidf.transform([query_text])
    tfidf_scores = (tfidf_matrix[idx] @ q_tfidf.T).toarray().ravel()

    role_text = parsed.get("role","")
    skill_text = " ".join(parsed.get("technical_skills",[]))
    comp_text = " ".join(parsed.get("competencies",[]))

    role_emb = model.encode(role_text, normalize_embeddings=True) if role_text else None
    skill_emb = model.encode(skill_text, normalize_embeddings=True) if skill_text else None
    comp_emb = model.encode(comp_text, normalize_embeddings=True) if comp_text else None

    role_sim = cosine_similarity([role_emb], embeddings[idx])[0] if role_emb is not None else np.zeros(len(idx))
    skill_sim = cosine_similarity([skill_emb], embeddings[idx])[0] if skill_emb is not None else np.zeros(len(idx))
    comp_sim = cosine_similarity([comp_emb], embeddings[idx])[0] if comp_emb is not None else np.zeros(len(idx))

    if intent == "skill":

        score = (
            0.45*skill_sim +
            0.25*sims[idx] +
            0.15*tfidf_scores +
            0.15*comp_sim
        )

    elif intent == "role":

        score = (
            0.45*role_sim +
            0.25*sims[idx] +
            0.15*tfidf_scores +
            0.15*comp_sim
        )

    elif intent == "competency":

        score = (
            0.45*comp_sim +
            0.25*sims[idx] +
            0.15*tfidf_scores +
            0.15*role_sim
        )

    else:

        score = (
            0.4*sims[idx] +
            0.3*tfidf_scores +
            0.3*comp_sim
        )

    return score

In [21]:
def parse_experience(exp):

    if exp is None:
        return None

    # If already numeric
    if isinstance(exp, (int, float)):
        return float(exp)

    exp = str(exp).lower()

    # extract numbers
    nums = re.findall(r"\d+\.?\d*", exp)

    if not nums:
        return None

    nums = [float(x) for x in nums]

    # if range like 2-4 years → use midpoint
    if len(nums) > 1:
        return sum(nums) / len(nums)

    return nums[0]


def experience_boost(df_subset, query_exp):

    query_exp = parse_experience(query_exp)

    if query_exp is None:
        return np.zeros(len(df_subset))

    boosts = []

    for exp in df_subset.get("experience_numeric", []):

        try:
            midpoint = exp.get("midpoint", None)
        except:
            midpoint = None

        if midpoint is None:
            boosts.append(0)
            continue

        dist = abs(midpoint - query_exp)

        # smooth decay
        score = max(0, 0.1 - 0.02 * dist)

        boosts.append(score)

    return np.array(boosts)

In [14]:
def ensure_coverage(results, competencies):

    if not competencies:
        return results.head(10)

    selected = []
    covered = set()

    for idx,row in results.iterrows():

        text = row["retrieval_text"].lower()

        matches = [c for c in competencies if c.lower() in text]

        if matches:

            selected.append(idx)
            covered.update(matches)

        if len(selected) >= 10:
            break

    for idx in results.index:

        if idx not in selected:

            selected.append(idx)

        if len(selected) >= 10:
            break

    return results.loc[selected].head(10)

In [15]:
def recommend(query):

    parsed = reason_query(query)

    intent = detect_intent(parsed)

    query_text = build_query_text(parsed)

    idx, sims = retrieve_candidates(query_text)

    scores = adaptive_score(parsed, intent, idx, sims, query_text)

    res = df.iloc[idx].copy()

    res["score"] = scores

    exp_boost = experience_boost(res, parsed.get("experience_years"))

    res["score"] += exp_boost

    res = res.sort_values("score", ascending=False)

    res = ensure_coverage(res, parsed.get("competencies",[]))

    return res

In [16]:
def slug(url):

    if not isinstance(url,str):
        return ""

    url = url.lower()

    m = re.search(r"/view/([^/]+)", url)

    if m:
        return m.group(1)

    return url

In [17]:
def recall_at_10(pred, true):

    p = {slug(u) for u in pred}
    t = {slug(u) for u in true}

    if not t:
        return 0

    return len(p & t) / len(t)

In [18]:
eval_df = pd.read_excel("Gen_AI Dataset.xlsx", sheet_name="Train-Set")

grouped = eval_df.groupby("Query")["Assessment_url"].apply(list).to_dict()

In [19]:
def evaluate():

    recalls = []

    for q, urls in tqdm(grouped.items()):

        res = recommend(q)

        preds = res["url"].tolist()

        r = recall_at_10(preds, urls)

        recalls.append(r)

    return np.mean(recalls)

In [24]:
mean_recall = evaluate()

print("Mean Recall@10:", mean_recall)

100%|██████████| 10/10 [00:33<00:00,  3.31s/it]

Mean Recall@10: 0.08


In [25]:
rows = []

for query, true_urls in grouped.items():

    true_slugs = {slug(u) for u in true_urls}

    res = recommend(query)

    for rank, row in enumerate(res.itertuples(), start=1):

        pred_slug = slug(row.url)

        rows.append({
            "query": query,
            "rank": rank,
            "assessment_name": row.name,
            "predicted_url": row.url,
            "score": row.score,
            "predicted_slug": pred_slug,
            "is_relevant": pred_slug in true_slugs,
            "ground_truth_urls": ", ".join(true_urls)
        })

df_compare = pd.DataFrame(rows)

df_compare.head()

,query,rank,assessment_name,predicted_url,score,predicted_slug,is_relevant,ground_truth_urls
0,Based on the JD below recommend me assessment ...,1,Time Management (U.S.),https://www.shl.com/products/product-catalog/v...,0.340450,time-management-u-s,False,https://www.shl.com/solutions/products/product...
1,Based on the JD below recommend me assessment ...,2,Managerial Scenarios Narrative Report,https://www.shl.com/products/product-catalog/v...,0.318454,managerial-scenarios-narrative-report,False,https://www.shl.com/solutions/products/product...
2,Based on the JD below recommend me assessment ...,3,Managerial Scenarios Candidate Report,https://www.shl.com/products/product-catalog/v...,0.318454,managerial-scenarios-candidate-report,False,https://www.shl.com/solutions/products/product...
3,Based on the JD below recommend me assessment ...,4,Managerial Scenarios Profile Report,https://www.shl.com/products/product-catalog/v...,0.318454,managerial-scenarios-profile-report,False,https://www.shl.com/solutions/products/product...
4,Based on the JD below recommend me assessment ...,5,OPQ Universal Competency Report 1.0,https://www.shl.com/products/product-catalog/v...,0.282232,opq-universal-competency-report,False,https://www.shl.com/solutions/products/product...


In [26]:
df_compare.to_excel("recommendation_comparison_5.xlsx", index=False)

print("Saved to recommendation_comparison.xlsx")

Saved to recommendation_comparison.xlsx
